In [1]:
##modules
#%matplotlib widget
#%matplotlib inline
#
%matplotlib qt
import mne
import numpy as np
import matplotlib
# Establecer un backend interactivo, como 'Qt5Agg', 'GTK3Agg', etc.
# Esto depende de los backends disponibles en tu sistema.

import matplotlib.pyplot as plt

#matplotlib.use('TkAgg')  # Asegúrate de que este backend está instalado.

import pandas as pd 
import os
import sys

from mne.preprocessing import ICA, corrmap, create_ecg_epochs, create_eog_epochs

from os.path import join as pathjoin
from time import time

from pathlib import Path

from autoreject import AutoReject

# aplicar la acf EN epochs

from statsmodels.tsa.stattools import acf
import numpy as np

import pandas as pd

In [2]:
sys.path.append("..")  # esto sube un nivel desde Scripts_visual_block
# --- Configuración dinámica de rutas ---
from get_paths_MOUS import get_paths_MOUS

# Parámetros editables
disco = "g"
modality = "visual"
layer_script = "block"
subj = "sub-V1001"

# Generar variables automáticamente
path_dict = get_paths_MOUS(disco=disco, modality=modality, layer_script=layer_script, subj=subj)
globals().update(path_dict)

# Mostrar todos los paths generados
print("\n📁 Rutas generadas:")
for k, v in path_dict.items():
    print(f"{k:<20} → {v}")

mne.utils.set_config('SUBJECTS_DIR', r'\\wsl$\Ubuntu-20.04\usr\local\freesurfer\subjects', set_env=True)


📁 Rutas generadas:
datadir              → g:\MOUS_204\MOUS_visual
general_datadir      → g:\MOUS_204
mri_dir              → g:\MOUS_204\sub-V1001\anat
meg_dir              → g:\MOUS_204\sub-V1001\meg
preproc_path         → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block
channels_structure_path → g:\MOUS_204\channels_structure
epochs_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_block
ICA_path             → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\ICA_block
epochs_clean_path    → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block
evoked_path          → g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\evoked_block
source_path          → g:\MOUS_204\MOUS_visual\output_source\source_block
raw_hsp_path         → g:\MOUS_204\MOUS_visual\output_source\source_block\raw_hsp
fwd_path             → g:\MOUS_204\MOUS_visual\output_source\source_block\fwd
inverse_path         → g:\MOUS_204\MOUS_visual\output_source\source_block

# LECTURA EPOCHS

In [3]:
epochs_woorden=mne.read_epochs(epochs_clean_path / f"{subj}_epochs_woorden_{layer_script}-epo.fif")

Reading g:\MOUS_204\MOUS_visual\output_preproc\preproc_block\epochs_clean_block\sub-V1001_epochs_woorden_block-epo.fif ...
    Found the data of interest:
        t =       0.00 ...   60000.00 ms
        0 CTF compensation matrices available
Not setting metadata
20 matching events found
No baseline correction applied
0 projection items activated


In [4]:

#se coge este sujeto, entiendo porque tiene los 272 canales comunes a todos los sujetos
epochs_mag_clean = epochs_woorden.copy().pick(picks="mag", exclude='bads')

print(len(epochs_mag_clean.ch_names))  # Esto debería darte 272

# Canales efectivos reales del objeto epochs (ya vienen con -4304)
canales_efectivos = set(epochs_mag_clean.ch_names)  # conjunto para acceso rápido



# Obtenemos matriz de adjacencia general, y el total de los canales
adjacency, ch_names_total = mne.channels.find_ch_adjacency(epochs_woorden.info, ch_type='mag')
print(adjacency.shape)  # Esto debería darte (272, 272)

# Convertimos a string normal por si acaso (np.str_ a str)
ch_names_total_str = [str(ch) for ch in ch_names_total]

# Creamos la tabla de los canales con sus nombres y sufijos
df_canales = pd.DataFrame({
    'indice': range(len(ch_names_total_str)),
    'nombre_base': ch_names_total_str,
    'nombre_con_sufijo': [f"{ch}-4304" for ch in ch_names_total_str]
})

# Mostramos las primeras filas
print(df_canales.head())


# Añadimos la columna 'canal_efectivo': si está en el set, lo ponemos, si no, NaN
df_canales[f"canal_efectivo_{modality}"] = df_canales['nombre_con_sufijo'].apply(
    lambda ch: ch if ch in canales_efectivos else np.nan
)

indices_efectivos = df_canales.loc[df_canales[f'canal_efectivo_{modality}'].notna(), 'indice'].to_list()

# Paso 2: Recortar la matriz de adyacencia original
from scipy.sparse import csr_matrix

# adjacency_total es sparse, así que podemos hacer slicing con arrays de índices
adjacency_reducida = adjacency[indices_efectivos, :][:, indices_efectivos]

# Confirmamos la forma
print(f"✅ adjacency_reducida creada con forma: {adjacency_reducida.shape}")

273
Reading adjacency matrix for ctf275.
(275, 275)
   indice nombre_base nombre_con_sufijo
0       0       MLC11        MLC11-4304
1       1       MLC12        MLC12-4304
2       2       MLC13        MLC13-4304
3       3       MLC14        MLC14-4304
4       4       MLC15        MLC15-4304
✅ adjacency_reducida creada con forma: (273, 273)


In [5]:
# Nombre completo del archivo
csv_path = os.path.join(channels_structure_path, f"channels_mag_{modality}.csv")

# Guardar el DataFrame
df_canales.to_csv(csv_path, index=False)

print(f"✅ Archivo guardado como: {csv_path}")

✅ Archivo guardado como: g:\MOUS_204\channels_structure\channels_mag_visual.csv


In [7]:
####lectura de la matriz de adyacencia (tiene que ir despue por fuerza)

import pickle
with open(channels_structure_path / f"adjacency_reduced_{modality}.pkl", "wb") as f:
    pickle.dump(adjacency_reducida, f)


📁 Guardando en: g:\apuntes_mne\code_MOUS\analysis
